In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    SGDRegressor
)
from sklearn.metrics import (
    mean_squared_error, r2_score,
)
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
df_reg = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
df_reg.head()

# Pre- Processing

In [ ]:
df_reg.describe()

In [ ]:
df_reg.info()

In [ ]:
# Drop a specific column by name
df_reg = df_reg.drop(columns=['Id'])

## Missin value handle

In [ ]:
# 1. Define your threshold (e.g., drop if > 50% is missing)
threshold = 0.4

# 2. Calculate the percentage of missing values per column
null_pct = df_reg.isnull().mean()

# 3. Identify columns to drop
cols_to_drop = null_pct[null_pct > threshold].index

# 4. Drop them
df_reg = df_reg.drop(columns=cols_to_drop)

print(f"Dropped columns: {list(cols_to_drop)}")

In [ ]:
# Identify numerical columns (integers and floats)
num_cols = df_reg.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Identify categorical columns (objects and category types)
cat_cols = df_reg.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical: {num_cols}")
print(f"Categorical: {cat_cols}")

In [ ]:
# Identify categorical columns first
cat_cols = df_reg.select_dtypes(include=['object']).columns

# Print unique values for each
for col in cat_cols:
    print(f"--- {col} ---")
    print(df_reg[col].unique())

In [ ]:
# 1. Identify columns by type
float_cols = df_reg.select_dtypes(include=['float64']).columns
int_cols = df_reg.select_dtypes(include=['int64']).columns
obj_cols = df_reg.select_dtypes(include=['object']).columns

# 2. Apply Mean to Floats
for col in float_cols:
    df_reg[col] = df_reg[col].fillna(df_reg[col].mean())

# 3. Apply Median to Ints (The most suitable central measure for integers)
for col in int_cols:
    df_reg[col] = df_reg[col].fillna(df_reg[col].median())

# 4. Apply Mode (Max Frequency) to Objects
for col in obj_cols:
    # mode() returns a Series, so we take the first element [0]
    if not df_reg[col].mode().empty:
        df_reg[col] = df_reg[col].fillna(df_reg[col].mode()[0])

print("Null values handled.")

In [ ]:
df_reg.info()

## Remove highly correlated column

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# 1. Isolate Numerical Columns
df_num = df_reg.select_dtypes(include=[np.number])

# 2. Calculate and Show Matrix
num_corr = df_num.corr().abs()
plt.figure(figsize=(20, 12))
sns.heatmap(num_corr, annot=True, cmap='Blues')
plt.title("Numerical Correlation Matrix")
plt.show()

# 3. Identify and Remove
upper = num_corr.where(np.tril(np.ones(num_corr.shape), k=-1).astype(bool))
to_drop_num = [col for col in upper.columns if any(upper[col] > 0.75)]
df_reg = df_reg.drop(columns=to_drop_num)

print(f"Dropped Numerical: {to_drop_num}")
# as the target column was dropping when dropping the last

In [ ]:
# 1. Isolate Categorical Columns
df_cat = df_reg.select_dtypes(include=['object']).copy()

# 2. Convert text to temporary codes for correlation check
for col in df_cat.columns:
    df_cat[col] = df_cat[col].astype('category').cat.codes

# 3. Calculate and Show Matrix
cat_corr = df_cat.corr().abs()
plt.figure(figsize=(25, 12))
sns.heatmap(cat_corr, annot=True, cmap='Greens')
plt.title("Categorical Correlation Matrix")
plt.show()

# 4. Identify and Remove
upper_cat = cat_corr.where(np.triu(np.ones(cat_corr.shape), k=1).astype(bool))
to_drop_cat = [col for col in upper_cat.columns if any(upper_cat[col] > 0.75)]
df_reg = df_reg.drop(columns=to_drop_cat)

print(f"Dropped Categorical: {to_drop_cat}")

## Drop Outlier

In [ ]:
# 1. Select integer columns
int_cols = df_reg.select_dtypes(include=['int64','float64'])

# 2. Get min and max, then transpose
summary = int_cols.describe().loc[['min', 'max']].T

# 3. Add a calculated Range column
summary['range'] = summary['max'] - summary['min']

# 4. Rename columns for better headers
summary.columns = ['Minimum', 'Maximum', 'Range']

print("--- Integer Column Ranges ---")
print(summary)

In [ ]:
# 1. Identify numerical columns
num_cols = df_reg.select_dtypes(include=[np.number]).columns

# Record the starting number of rows
initial_rows = len(df_reg)
print(f"Total rows in dataset before cleaning: {initial_rows}")
print("-" * 30)

for col in num_cols:
    rows_before = len(df_reg)
    
    # 2. Calculate IQR
    Q1 = df_reg[col].quantile(0.15)
    Q3 = df_reg[col].quantile(0.85)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # 3. Filter (Drop the outliers)
    df_reg = df_reg[(df_reg[col] >= lower) & (df_reg[col] <= upper)]
    
    rows_after = len(df_reg)
    rows_dropped = rows_before - rows_after
    
    if rows_dropped > 0:
        print(f"Column '{col}': Dropped {rows_dropped} rows")

# 4. Final Summary
final_rows = len(df_reg)
total_dropped = initial_rows - final_rows
print("-" * 30)
print(f"Cleaning Complete.")
print(f"Final row count: {final_rows}")
print(f"Total rows removed: {total_dropped} ({(total_dropped/initial_rows)*100:.2f}% of data)")

## Train-test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [ ]:
# 1. Separate Features and Target
X = df_reg.drop(columns=['SalePrice'])
y = df_reg['SalePrice']

# 2. Simple Train-Test Split (no stratification)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Identify Column Types
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object']).columns

# 4. Create the Transformer
# This applies scaling to numbers and one-hot encoding to categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])

# 5. Fit on Train, Transform both (Prevents Data Leakage)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
X_train_r = X_train_transformed
X_test_r = X_test_transformed
y_train_r = y_train
y_test_r = y_test

In [ ]:
# Regression Evaluation Function
def evaluate_regression(model):
    model.fit(X_train_r, y_train_r)
    y_pred = model.predict(X_test_r)
    
    print("R2 Score:", r2_score(y_test_r, y_pred))
    print("MSE:", mean_squared_error(y_test_r, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test_r, y_pred)))
    print("-"*40)

# Multiple Linear Regression (MLR)


In [ ]:
lr = LinearRegression()
evaluate_regression(lr)

# Ridge Regression

In [ ]:
# Ridge Regression
ridge = Ridge(alpha=1.0)
evaluate_regression(ridge)

# Lasso Regression

In [ ]:
# Lasso Regression
lasso = Lasso(alpha=0.1)
evaluate_regression(lasso)

# ElasticNet

In [ ]:
# ElasticNet
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
evaluate_regression(elastic)

# Polynomial Regression

In [ ]:
# Polynomial Regression
poly = PolynomialFeatures(degree=2)
X_poly_train = poly.fit_transform(X_train_r)
X_poly_test = poly.transform(X_test_r)

poly_model = LinearRegression()
poly_model.fit(X_poly_train, y_train_r)

y_pred_poly = poly_model.predict(X_poly_test)

print("Polynomial R2:", r2_score(y_test_r, y_pred_poly))

# SGD Regressor

In [ ]:
# SGD Regressor
sgd_reg = SGDRegressor(max_iter=1000, learning_rate='constant', eta0=0.001)
evaluate_regression(sgd_reg)

# KNN

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# n_neighbors=5 is the standard starting point
knn_reg = KNeighborsRegressor(n_neighbors=5, weights='uniform')

# Evaluate using your custom function
evaluate_regression(knn_reg)

# SVM

In [ ]:
from sklearn.svm import SVR

# C controls regularization; epsilon defines the "margin of error"
svm_reg = SVR(kernel='rbf', C=100000, epsilon=0.1)

# Evaluate using your custom function
evaluate_regression(svm_reg)

# XGBoost

In [ ]:
from xgboost import XGBRegressor

# Initialize the model
# n_estimators: number of trees
# max_depth: how deep each tree can go
# learning_rate: step size shrinkage to prevent overfitting
xgb_reg = XGBRegressor(
    n_estimators=100, 
    max_depth=7, 
    learning_rate=0.1, 
    random_state=42,
    n_jobs=-1 # Uses all available CPU cores
)

# Evaluate using your custom function
evaluate_regression(xgb_reg)